Sempre que for utilizar a simulação, fazer o upload dos arquivos do certificado do IoT Core e instalação do paho-mqtt


In [6]:
!pip install paho-mqtt

In [7]:
import paho.mqtt.client as mqtt
import json
import time
import random
import ssl
from datetime import datetime

In [8]:
# ==============================================================================
# CONFIGURAÇÕES AWS IOT CORE
# ==============================================================================
# Substitua pelo seu endpoint do AWS IoT Core (encontrado em Settings no console)
AWS_ENDPOINT = "arl4q7tru8xla-ats.iot.us-east-1.amazonaws.com"
PORT = 8883 # Porta padrão para MQTT com TLS na AWS
SUBESTACAO_ID = "SUB-01"

# Caminho dos certificados (faça upload no Colab e ajuste os nomes)
ROOT_CA_PATH = "AmazonRootCA1.pem"
CERT_PATH = "device-certificate.pem.crt"
KEY_PATH = "device-private.pem.key"

# Tópicos
TOPIC_ACESSO = f"subestacao/{SUBESTACAO_ID}/acesso/request"
TOPIC_PORTA = f"subestacao/{SUBESTACAO_ID}/sensor/porta"
TOPIC_PRESENCA = f"subestacao/{SUBESTACAO_ID}/sensor/presenca"
TOPIC_AMBIENTE = f"subestacao/{SUBESTACAO_ID}/sensor/ambiente" # Novo tópico

In [9]:
# ==============================================================================
# FUNÇÕES DE SIMULAÇÃO
# ==============================================================================
def simular_leitura_rfid():
    # Gera uma lista de strings de "1" a "100"
    uids = [str(i) for i in range(1, 101)]

    return {
        "rfid_uid": random.choice(uids),
        "leitor_id": "RFID-PORTAO-01",
        "nivel_bateria_leitor_percent": round(random.uniform(80.0, 100.0), 1),
        "latencia_leitura_ms": random.randint(15, 45),
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

def simular_sensor_porta():
    return {
        "porta_id": "PORTA-PRINCIPAL",
        "status": random.choice(["ABERTA", "FECHADA"]),
        "tempo_permanencia_aberta_s": random.randint(0, 120),
        "tensao_trava_v": round(random.uniform(23.5, 24.5), 2), # Trava de 24V
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

def simular_sensor_presenca():
    return {
        "zona_id": "ZONA-ALTA-TENSAO",
        "detectado": True,
        "nivel_confianca_percent": round(random.uniform(90.0, 99.9), 1),
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

def simular_sensor_ambiente():
    # Novo payload ideal para séries temporais no InfluxDB
    return {
        "temperatura_c": round(random.uniform(25.0, 45.0), 2),
        "umidade_percent": round(random.uniform(30.0, 70.0), 1),
        "fator_potencia": round(random.uniform(0.92, 0.99), 2),
        "qualidade_sinal_wifi_dbm": random.randint(-85, -40),
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

In [10]:
# ==============================================================================
# CONFIGURAÇÃO MQTT
# ==============================================================================
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("✅ Conectado ao AWS IoT Core com sucesso!")
    else:
        print(f"❌ Falha na conexão. Código: {rc}")

client = mqtt.Client(client_id="Gateway-ESP32-Simulador")
client.on_connect = on_connect

# Configuração do TLS para AWS
client.tls_set(ca_certs=ROOT_CA_PATH, certfile=CERT_PATH, keyfile=KEY_PATH, cert_reqs=ssl.CERT_REQUIRED, tls_version=ssl.PROTOCOL_TLSv1_2, ciphers=None)

print("Tentando conectar à AWS...")
client.connect(AWS_ENDPOINT, PORT, 60)
client.loop_start()

try:
    for i in range(15):
        time.sleep(2)
        evento = random.choice(["rfid", "porta", "presenca", "ambiente"])

        if evento == "rfid":
            dados = simular_leitura_rfid()
            client.publish(TOPIC_ACESSO, json.dumps(dados), qos=1)
            print(f"🔑 [RFID] Payload: {dados}")

        elif evento == "porta":
            dados = simular_sensor_porta()
            client.publish(TOPIC_PORTA, json.dumps(dados), qos=1)
            print(f"🚪 [PORTA] Payload: {dados}")

        elif evento == "presenca":
            dados = simular_sensor_presenca()
            client.publish(TOPIC_PRESENCA, json.dumps(dados), qos=1)
            print(f"🚶 [PRESENÇA] Payload: {dados}")

        elif evento == "ambiente":
            dados = simular_sensor_ambiente()
            client.publish(TOPIC_AMBIENTE, json.dumps(dados), qos=1)
            print(f"🌡️ [AMBIENTE] Payload: {dados}")

except KeyboardInterrupt:
    print("\n🛑 Simulação interrompida.")
finally:
    client.loop_stop()
    client.disconnect()
    print("🔌 Desconectado.")

/tmp/ipykernel_8267/3796008469.py:10: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="Gateway-ESP32-Simulador")


Tentando conectar à AWS...
✅ Conectado ao AWS IoT Core com sucesso!


/tmp/ipykernel_8267/1885543209.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🔑 [RFID] Payload: {'rfid_uid': '57', 'leitor_id': 'RFID-PORTAO-01', 'nivel_bateria_leitor_percent': 95.4, 'latencia_leitura_ms': 44, 'timestamp': '2026-06-09T18:40:06.805573Z'}


/tmp/ipykernel_8267/1885543209.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'FECHADA', 'tempo_permanencia_aberta_s': 83, 'tensao_trava_v': 23.85, 'timestamp': '2026-06-09T18:40:08.809559Z'}
🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'FECHADA', 'tempo_permanencia_aberta_s': 96, 'tensao_trava_v': 23.56, 'timestamp': '2026-06-09T18:40:10.810066Z'}


/tmp/ipykernel_8267/1885543209.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🚶 [PRESENÇA] Payload: {'zona_id': 'ZONA-ALTA-TENSAO', 'detectado': True, 'nivel_confianca_percent': 95.9, 'timestamp': '2026-06-09T18:40:12.810938Z'}


/tmp/ipykernel_8267/1885543209.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


🌡️ [AMBIENTE] Payload: {'temperatura_c': 43.47, 'umidade_percent': 65.9, 'fator_potencia': 0.92, 'qualidade_sinal_wifi_dbm': -80, 'timestamp': '2026-06-09T18:40:14.811741Z'}
🔑 [RFID] Payload: {'rfid_uid': '4', 'leitor_id': 'RFID-PORTAO-01', 'nivel_bateria_leitor_percent': 86.7, 'latencia_leitura_ms': 45, 'timestamp': '2026-06-09T18:40:16.813014Z'}
🌡️ [AMBIENTE] Payload: {'temperatura_c': 25.24, 'umidade_percent': 49.1, 'fator_potencia': 0.95, 'qualidade_sinal_wifi_dbm': -84, 'timestamp': '2026-06-09T18:40:18.814086Z'}
🔑 [RFID] Payload: {'rfid_uid': '60', 'leitor_id': 'RFID-PORTAO-01', 'nivel_bateria_leitor_percent': 91.1, 'latencia_leitura_ms': 30, 'timestamp': '2026-06-09T18:40:20.815301Z'}
🚪 [PORTA] Payload: {'porta_id': 'PORTA-PRINCIPAL', 'status': 'ABERTA', 'tempo_permanencia_aberta_s': 64, 'tensao_trava_v': 23.89, 'timestamp': '2026-06-09T18:40:22.815966Z'}
🔑 [RFID] Payload: {'rfid_uid': '100', 'leitor_id': 'RFID-PORTAO-01', 'nivel_bateria_leitor_percent': 89.8, 'latencia_leitura_